# Static UMAP Plotting by Time Since Merger

This notebook plots static UMAP panels where the point color is the raw time since merger value from the TNG Tools catalog. It mirrors `static_umap_plotting_w_xmatched_samples_single_flag_raw_merger_flags.ipynb`, but uses continuous color instead of time-window threshold overlays.

The default example is `run8`, experiments `9-13`, with one row per experiment and three horizontal panels for `Mini`, `Minor`, and `Major` mergers.

## Workflow

1. Resolve the Hyrax run log and config for each run/experiment pair.
2. Load the matching UMAP coordinates and object IDs from Hyrax metadata.
3. Load the TNG Tools `catalog.fits` file with raw merger columns.
4. Match catalog rows to UMAP points by normalized object ID.
5. Plot all UMAP points in gray and color matched merger points by `*_TimeSinceMerger`.

The no-merger sentinel is usually `-1`, so the default valid range is `time_since_merger >= 0`. Use `max_time_gyr=3.0` if you want to restrict the coloring to recent mergers only.

## Imports

In [ ]:
from __future__ import annotations

import logging
import re
from pathlib import Path
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
import hyrax
from IPython.display import display

## Paths, Catalog Defaults, and Merger Columns

Adjust these paths if your run outputs live somewhere else. The catalog paths should point to the TNG Tools `catalog.fits` outputs that include `Mini_TimeSinceMerger`, `Minor_TimeSinceMerger`, and `Major_TimeSinceMerger`.

In [ ]:
def first_existing_path(candidates: Iterable[str | Path]) -> Path:
    """Return the first existing path, or the first candidate for readable errors."""
    paths = [Path(candidate).expanduser() for candidate in candidates]
    for path in paths:
        if path.exists():
            return path
    return paths[0]


PROJECT_ROOT = Path.cwd()

HYRAX_RUN_BASE = first_existing_path([
    '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs',
    '/mmfs1/gscratch/dirac/aritrag/hyrax_comcam_dp1_runs',
    PROJECT_ROOT / 'hyrax_runs',
    PROJECT_ROOT / 'Hyrax-Research' / 'hyrax_runs',
])

CATALOG_PATHS = {
    'all': first_existing_path([
        '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog.fits',
        PROJECT_ROOT / 'data' / 'catalog.fits',
        PROJECT_ROOT / 'Hyrax-Research' / 'data' / 'catalog.fits',
    ]),
    'le_120x120': first_existing_path([
        '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_le_120x120.fits',
        PROJECT_ROOT / 'data' / 'catalog_le_120x120.fits',
        PROJECT_ROOT / 'Hyrax-Research' / 'data' / 'catalog_le_120x120.fits',
    ]),
    'gt_120x120': first_existing_path([
        '/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_gt_120x120.fits',
        PROJECT_ROOT / 'data' / 'catalog_gt_120x120.fits',
        PROJECT_ROOT / 'Hyrax-Research' / 'data' / 'catalog_gt_120x120.fits',
    ]),
}

DEFAULT_CATALOG_KEY = 'all'

# Optional: add only exceptional run-specific defaults here.
# Example: RUN_CATALOG_OVERRIDES = {12: 'le_120x120', 13: 'gt_120x120'}
RUN_CATALOG_OVERRIDES = {}

MERGER_TYPES = ('Mini', 'Minor', 'Major')
TIME_SINCE_MERGER_COLUMNS = {
    merger_type: f'{merger_type}_TimeSinceMerger'
    for merger_type in MERGER_TYPES
}


def normalize_merger_type(merger_type: str) -> str:
    """Normalize merger type labels to the catalog column convention."""
    normalized = merger_type.strip().capitalize()
    if normalized not in TIME_SINCE_MERGER_COLUMNS:
        valid = ', '.join(TIME_SINCE_MERGER_COLUMNS)
        raise KeyError(f"Unknown merger type '{merger_type}'. Choose one of: {valid}")
    return normalized


def time_since_merger_columns(merger_types: Sequence[str] = MERGER_TYPES) -> list[str]:
    """Return raw time-since-merger catalog columns for the requested merger types."""
    return [TIME_SINCE_MERGER_COLUMNS[normalize_merger_type(kind)] for kind in merger_types]


def resolve_catalog_key(run: int, catalog_key: str | None = None) -> str:
    """Resolve the catalog choice for a run, defaulting all runs to the full catalog."""
    resolved = catalog_key or RUN_CATALOG_OVERRIDES.get(run, DEFAULT_CATALOG_KEY)
    if resolved not in CATALOG_PATHS:
        valid = ', '.join(sorted(CATALOG_PATHS))
        raise KeyError(f"Unknown catalog_key '{resolved}'. Choose one of: {valid}")
    return resolved

## Catalog Loading and ID Normalization

In [ ]:
def load_external_catalog(catalog_path: str | Path) -> pd.DataFrame:
    """Load an image/sample catalog from FITS, parquet, or CSV."""
    catalog_path = Path(catalog_path)
    suffix = catalog_path.suffix.lower()

    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(catalog_path)

    if suffix in {'.csv', '.txt'}:
        return pd.read_csv(catalog_path)

    if suffix in {'.fits', '.fit', '.fts'}:
        from astropy.table import Table

        return Table.read(catalog_path).to_pandas()

    raise ValueError(f"Unsupported catalog format '{suffix}'. Use FITS, parquet, or CSV.")


def _decode_scalar(value):
    """Decode byte strings from FITS tables while leaving other values unchanged."""
    if isinstance(value, (bytes, bytearray)):
        return value.decode('utf-8').strip()
    return value


def normalize_object_ids(values) -> pd.Series:
    """Normalize object IDs to comparable nullable strings."""
    ids = pd.Series(values, copy=False).map(_decode_scalar)
    missing = ids.isna()
    numeric = pd.to_numeric(ids, errors='coerce')

    if (~missing).any() and numeric.loc[~missing].notna().all():
        normalized = numeric.astype('Int64').astype('string')
    else:
        normalized = ids.astype('string').str.strip()

    return normalized.mask(missing)


def resolve_catalog_id_column(catalog: pd.DataFrame, catalog_id_column: str | None = None) -> str:
    """Find the catalog object-ID column used to match rows back to UMAP metadata."""
    candidates = [
        catalog_id_column,
        'object_id',
        'rubin_object_id',
        'objectId',
        'objectId_data',
        'id',
    ]

    for candidate in candidates:
        if candidate is not None and candidate in catalog.columns:
            return candidate

    raise KeyError(
        'Could not find an object ID column in the catalog. '
        'Pass catalog_id_column explicitly.'
    )

## Raw Time-Since-Merger Columns

This section validates the continuous merger columns and summarizes how many catalog rows have valid time-since values. Values below `min_time_gyr` are excluded by default so `-1` no-merger sentinels do not get plotted as real times.

In [ ]:
def prepare_catalog_with_time_since_merger(
    catalog: pd.DataFrame,
    merger_types: Sequence[str] = MERGER_TYPES,
    catalog_id_column: str | None = None,
) -> pd.DataFrame:
    """Validate raw time-since columns and add a UMAP match key."""
    catalog = catalog.copy()
    raw_columns = time_since_merger_columns(merger_types)
    missing = [column for column in raw_columns if column not in catalog.columns]

    if missing:
        raise KeyError(
            'The loaded catalog is missing these raw time-since-merger columns: '
            f'{missing}. Check that CATALOG_PATHS points to the TNG Tools '
            'catalog with appended merger-catalog fields.'
        )

    catalog_id_column = resolve_catalog_id_column(catalog, catalog_id_column)
    catalog['_match_id'] = normalize_object_ids(catalog[catalog_id_column])
    return catalog


def valid_time_since_merger_mask(
    values: pd.Series,
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
) -> pd.Series:
    """Select finite time-since-merger values inside the requested range."""
    mask = values.notna() & np.isfinite(values)
    mask &= values >= min_time_gyr
    if max_time_gyr is not None:
        mask &= values <= max_time_gyr
    return mask


def summarize_time_since_merger_catalog(
    catalog: pd.DataFrame,
    merger_types: Sequence[str] = MERGER_TYPES,
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
) -> pd.DataFrame:
    """Return counts and basic time ranges for the requested merger columns."""
    rows = []
    for merger_type in merger_types:
        merger_type = normalize_merger_type(merger_type)
        column = TIME_SINCE_MERGER_COLUMNS[merger_type]
        values = pd.to_numeric(catalog[column], errors='coerce')
        valid = valid_time_since_merger_mask(
            values,
            min_time_gyr=min_time_gyr,
            max_time_gyr=max_time_gyr,
        )
        valid_values = values.loc[valid]
        rows.append({
            'merger_type': merger_type,
            'column': column,
            'catalog_rows': len(catalog),
            'catalog_rows_with_match_id': int(catalog['_match_id'].notna().sum()),
            'valid_time_rows': int(valid.sum()),
            'min_time_gyr': valid_values.min() if not valid_values.empty else np.nan,
            'median_time_gyr': valid_values.median() if not valid_values.empty else np.nan,
            'max_time_gyr': valid_values.max() if not valid_values.empty else np.nan,
        })
    return pd.DataFrame(rows)


def resolve_time_color_limits(
    catalog: pd.DataFrame,
    merger_types: Sequence[str] = MERGER_TYPES,
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
) -> tuple[float, float]:
    """Resolve one shared color scale for all merger-type panels."""
    selected_values = []
    for merger_type in merger_types:
        column = TIME_SINCE_MERGER_COLUMNS[normalize_merger_type(merger_type)]
        values = pd.to_numeric(catalog[column], errors='coerce')
        valid = valid_time_since_merger_mask(
            values,
            min_time_gyr=min_time_gyr,
            max_time_gyr=max_time_gyr,
        )
        if valid.any():
            selected_values.append(values.loc[valid])

    if selected_values:
        all_values = pd.concat(selected_values, ignore_index=True)
        resolved_vmin = float(all_values.min()) if vmin is None else float(vmin)
        resolved_vmax = float(all_values.max()) if vmax is None else float(vmax)
    else:
        resolved_vmin = 0.0 if vmin is None else float(vmin)
        resolved_vmax = 1.0 if vmax is None else float(vmax)

    if resolved_vmin == resolved_vmax:
        delta = 0.5 if resolved_vmin == 0 else abs(resolved_vmin) * 0.05
        resolved_vmin -= delta
        resolved_vmax += delta

    return resolved_vmin, resolved_vmax

## Loading UMAP Results

In [ ]:
def resolve_umap_paths(
    run: int,
    expt: int,
    run_base: str | Path = HYRAX_RUN_BASE,
) -> tuple[Path, Path]:
    """Read the run/expt log and return the UMAP results directory plus config file."""
    run_dir = Path(run_base) / f'run{run}'
    run_name = f'udb{run}_{expt}'
    log_path = run_dir / f'{run_name}.txt'
    config_path = run_dir / f'{run_name}.toml'

    if not log_path.exists():
        raise FileNotFoundError(f'UMAP output log not found: {log_path}')

    content = log_path.read_text()
    match = re.search(r'Saving UMAP results to (.+)', content)
    if match is None:
        raise ValueError(f'Could not find UMAP results directory in {log_path}')

    return Path(match.group(1).strip()), config_path


def get_umap_with_ids(
    config=None,
    input_dir: str | Path | None = None,
    suppress_logs: bool = True,
    id_field: str = 'objectId_data',
) -> dict:
    """Load UMAP coordinates and the object IDs used for catalog matching."""
    from hyrax.data_sets.inference_dataset import InferenceDataSet

    def extract_metadata_column(metadata_obj, field_name: str):
        """Extract one metadata field from dict, DataFrame, structured array, or array payloads."""
        if isinstance(metadata_obj, dict):
            if field_name in metadata_obj:
                return np.asarray(metadata_obj[field_name])
            if len(metadata_obj) == 1:
                return np.asarray(next(iter(metadata_obj.values())))
            return None

        if hasattr(metadata_obj, 'columns'):
            columns = list(metadata_obj.columns)
            if field_name in columns:
                return metadata_obj[field_name].to_numpy()
            if len(columns) == 1:
                return metadata_obj[columns[0]].to_numpy()
            return None

        dtype = getattr(metadata_obj, 'dtype', None)
        names = getattr(dtype, 'names', None)
        if names:
            if field_name in names:
                return np.asarray(metadata_obj[field_name])
            if len(names) == 1:
                return np.asarray(metadata_obj[names[0]])
            return None

        array = np.asarray(metadata_obj)
        if array.ndim == 1:
            return array
        if array.ndim == 2 and array.shape[1] == 1:
            return array[:, 0]
        return None

    if suppress_logs:
        logging.disable(logging.CRITICAL)

    umap_results = InferenceDataSet(config, results_dir=input_dir, verb='umap')

    logging.disable(logging.NOTSET)

    points = np.array([point.numpy() for point in umap_results])
    x, y = points[:, 0], points[:, 1]

    all_indices = list(range(len(umap_results)))
    available_fields = list(umap_results.metadata_fields())
    preferred_fields = [
        id_field,
        'objectId_data',
        'object_id_data',
        'objectId',
        'object_id',
        'rubin_object_id',
        'id',
    ]

    candidate_fields = []
    for field in preferred_fields:
        if field is not None and field not in candidate_fields:
            candidate_fields.append(field)

    candidate_fields = [field for field in candidate_fields if field in available_fields] + [
        field for field in candidate_fields if field not in available_fields
    ]

    attempts = []
    rubin_ids = None
    resolved_field = None

    for candidate in candidate_fields:
        try:
            metadata = umap_results.metadata(all_indices, [candidate])
            extracted = extract_metadata_column(metadata, candidate)
            if extracted is None:
                attempts.append(f'{candidate}: not found in metadata payload')
                continue
            if len(extracted) != len(umap_results):
                attempts.append(f'{candidate}: length mismatch')
                continue

            rubin_ids = np.asarray(extracted)
            resolved_field = candidate
            break
        except Exception as exc:
            attempts.append(f'{candidate}: {exc}')

    if rubin_ids is None:
        raise KeyError(
            'Could not extract object IDs from UMAP metadata. Attempts: '
            + '; '.join(attempts)
        )

    return {
        'x': x,
        'y': y,
        'rubin_ids': rubin_ids,
        'id_field': resolved_field,
        'umap_results': umap_results,
    }


def load_umap_for_run_expt(
    run: int,
    expt: int,
    run_base: str | Path = HYRAX_RUN_BASE,
    suppress_logs: bool = True,
) -> dict:
    """Load one run/expt pair using the Hyrax config referenced by the run log."""
    umap_dir, config_file = resolve_umap_paths(run, expt, run_base=run_base)

    if suppress_logs:
        logging.disable(logging.CRITICAL)
    h = hyrax.Hyrax(config_file=config_file)
    logging.disable(logging.NOTSET)

    return get_umap_with_ids(
        config=h.config,
        input_dir=umap_dir,
        suppress_logs=suppress_logs,
    )

## Matching Catalog Times to UMAP Points

In [ ]:
def build_umap_lookup(umap_data: dict) -> pd.DataFrame:
    """Build the UMAP coordinate table keyed by normalized object ID."""
    return pd.DataFrame({
        'x': umap_data['x'],
        'y': umap_data['y'],
        '_match_id': normalize_object_ids(umap_data['rubin_ids']),
    }).dropna(subset=['_match_id'])


def match_time_since_merger_to_umap(
    umap_data: dict,
    catalog_with_times: pd.DataFrame,
    merger_type: str,
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
) -> pd.DataFrame:
    """Return UMAP coordinates with one continuous time-since-merger value."""
    merger_type = normalize_merger_type(merger_type)
    column = TIME_SINCE_MERGER_COLUMNS[merger_type]
    if column not in catalog_with_times.columns:
        raise KeyError(
            f"Raw merger column '{column}' not found. Available columns include: "
            f"{list(catalog_with_times.columns[:20])}"
        )

    values = pd.to_numeric(catalog_with_times[column], errors='coerce')
    selected = catalog_with_times.loc[
        valid_time_since_merger_mask(
            values,
            min_time_gyr=min_time_gyr,
            max_time_gyr=max_time_gyr,
        ),
        ['_match_id', column],
    ].copy()

    if selected.empty:
        return pd.DataFrame(columns=['x', 'y', '_match_id', 'time_since_merger_gyr'])

    selected = (
        selected
        .dropna(subset=['_match_id'])
        .drop_duplicates('_match_id')
        .rename(columns={column: 'time_since_merger_gyr'})
    )

    return selected.merge(build_umap_lookup(umap_data), on='_match_id', how='inner')

## Plotting Grids

These wrappers make one row per experiment and one column per merger type. The default color scale is shared across all panels so `Mini`, `Minor`, and `Major` are directly comparable.

In [ ]:
def plot_umap_time_since_merger_panel(
    ax,
    umap_data: dict,
    catalog_with_times: pd.DataFrame,
    merger_type: str,
    norm: Normalize,
    cmap: str = 'viridis',
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
    alpha_background: float = 0.35,
    s_background: float = 1,
    alpha_merger: float = 0.85,
    s_merger: float = 6,
    title: str | None = None,
    add_colorbar: bool = False,
) -> pd.DataFrame:
    """Plot one UMAP panel with points colored by time since merger."""
    merger_type = normalize_merger_type(merger_type)

    ax.scatter(
        umap_data['x'],
        umap_data['y'],
        alpha=alpha_background,
        s=s_background,
        c='lightgray',
        linewidths=0,
        rasterized=True,
    )

    matched = match_time_since_merger_to_umap(
        umap_data=umap_data,
        catalog_with_times=catalog_with_times,
        merger_type=merger_type,
        min_time_gyr=min_time_gyr,
        max_time_gyr=max_time_gyr,
    )

    scatter = None
    if not matched.empty:
        scatter = ax.scatter(
            matched['x'].to_numpy(),
            matched['y'].to_numpy(),
            c=matched['time_since_merger_gyr'].to_numpy(),
            cmap=cmap,
            norm=norm,
            alpha=alpha_merger,
            s=s_merger,
            linewidths=0,
            rasterized=True,
        )
    else:
        ax.text(
            0.5,
            0.5,
            'No matched merger points',
            ha='center',
            va='center',
            transform=ax.transAxes,
            fontsize='small',
        )

    if add_colorbar and scatter is not None:
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Time since merger [Gyr]')

    if title:
        ax.set_title(title)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')

    return matched


def plot_time_since_merger_grid(
    run: int,
    expts,
    catalog_with_times: pd.DataFrame,
    merger_types: Sequence[str] = MERGER_TYPES,
    run_base: str | Path = HYRAX_RUN_BASE,
    figsize: tuple | None = None,
    dpi: int = 150,
    save_path: str | Path | None = None,
    suptitle: str | None = None,
    suppress_logs: bool = True,
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    cmap: str = 'viridis',
    alpha_background: float = 0.35,
    s_background: float = 1,
    alpha_merger: float = 0.85,
    s_merger: float = 6,
    share_colorbar: bool = True,
):
    """Plot one row per experiment and one column per merger type."""
    try:
        from tqdm.notebook import tqdm
    except Exception:
        tqdm = lambda iterable, total=None: iterable

    expts = list(expts)
    merger_types = [normalize_merger_type(kind) for kind in merger_types]
    nrows = len(expts)
    ncols = len(merger_types)

    if nrows == 0:
        raise ValueError('At least one experiment is required.')
    if ncols == 0:
        raise ValueError('At least one merger type is required.')
    if figsize is None:
        figsize = (ncols * 4.2, nrows * 3.4)

    resolved_vmin, resolved_vmax = resolve_time_color_limits(
        catalog_with_times,
        merger_types=merger_types,
        min_time_gyr=min_time_gyr,
        max_time_gyr=max_time_gyr,
        vmin=vmin,
        vmax=vmax,
    )
    norm = Normalize(vmin=resolved_vmin, vmax=resolved_vmax)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        dpi=dpi,
        squeeze=False,
        constrained_layout=True,
    )

    matched_by_panel = {}

    for row, expt in enumerate(tqdm(expts, total=nrows)):
        try:
            umap_data = load_umap_for_run_expt(
                run,
                expt,
                run_base=run_base,
                suppress_logs=suppress_logs,
            )
        except Exception as exc:
            for col in range(ncols):
                ax = axes[row, col]
                ax.text(
                    0.5,
                    0.5,
                    f'Error loading\nRun {run}, Expt {expt}\n{exc}',
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                    fontsize='small',
                )
                ax.set_title(f'Run {run}, Expt {expt}')
            continue

        for col, merger_type in enumerate(merger_types):
            ax = axes[row, col]
            title = f'Run {run}, Expt {expt}: {merger_type}'

            try:
                matched = plot_umap_time_since_merger_panel(
                    ax=ax,
                    umap_data=umap_data,
                    catalog_with_times=catalog_with_times,
                    merger_type=merger_type,
                    norm=norm,
                    cmap=cmap,
                    min_time_gyr=min_time_gyr,
                    max_time_gyr=max_time_gyr,
                    alpha_background=alpha_background,
                    s_background=s_background,
                    alpha_merger=alpha_merger,
                    s_merger=s_merger,
                    title=title,
                    add_colorbar=not share_colorbar,
                )
                matched_by_panel[(expt, merger_type)] = matched
            except Exception as exc:
                ax.text(
                    0.5,
                    0.5,
                    str(exc),
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                    fontsize='small',
                )
                ax.set_title(title)

    if suptitle:
        fig.suptitle(suptitle, fontsize=16)

    if share_colorbar:
        mappable = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        mappable.set_array([])
        cbar = fig.colorbar(
            mappable,
            ax=axes.ravel().tolist(),
            shrink=0.88,
            pad=0.01,
        )
        cbar.set_label('Time since merger [Gyr]')

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    else:
        plt.show()

    return fig, axes, matched_by_panel


def plot_run_time_since_merger_grid(
    run: int,
    expts=range(9, 14),
    merger_types: Sequence[str] = MERGER_TYPES,
    catalog_key: str | None = None,
    catalog_id_column: str | None = None,
    run_base: str | Path = HYRAX_RUN_BASE,
    min_time_gyr: float = 0.0,
    max_time_gyr: float | None = None,
    **kwargs,
):
    """Load the selected catalog and plot continuous time-since-merger grids."""
    resolved_catalog_key = resolve_catalog_key(run, catalog_key=catalog_key)
    catalog = load_external_catalog(CATALOG_PATHS[resolved_catalog_key])
    catalog_with_times = prepare_catalog_with_time_since_merger(
        catalog=catalog,
        merger_types=merger_types,
        catalog_id_column=catalog_id_column,
    )

    display(summarize_time_since_merger_catalog(
        catalog_with_times,
        merger_types=merger_types,
        min_time_gyr=min_time_gyr,
        max_time_gyr=max_time_gyr,
    ))

    return plot_time_since_merger_grid(
        run=run,
        expts=expts,
        catalog_with_times=catalog_with_times,
        merger_types=merger_types,
        run_base=run_base,
        min_time_gyr=min_time_gyr,
        max_time_gyr=max_time_gyr,
        suptitle=f'Run {run} ({resolved_catalog_key}) time since merger from catalog.fits',
        **kwargs,
    )

## Quick Catalog Sanity Check

Run this before plotting if you want to verify that the resolved `catalog.fits` contains all three raw time-since-merger columns.

In [ ]:
catalog_key = 'all'
catalog = load_external_catalog(CATALOG_PATHS[catalog_key])
requested_columns = time_since_merger_columns(MERGER_TYPES)
available_columns = [column for column in requested_columns if column in catalog.columns]
missing_columns = [column for column in requested_columns if column not in catalog.columns]

print(CATALOG_PATHS[catalog_key])
print(f'Available time-since-merger columns: {available_columns}')
if missing_columns:
    print(f'Missing time-since-merger columns: {missing_columns}')

preview_columns = [resolve_catalog_id_column(catalog), *available_columns]
catalog[preview_columns].head()

## Example: Run 8, Experiments 9-13

This creates five rows, one for each experiment, and three horizontal panels per row: `Mini`, `Minor`, and `Major`. All valid merger points use the same shared color scale.

In [ ]:
fig, axes, matched_by_panel = plot_run_time_since_merger_grid(
    run=8,
    expts=range(9, 14),
    merger_types=('Mini', 'Minor', 'Major'),
    alpha_background=0.35,
    s_background=1,
    alpha_merger=0.85,
    s_merger=6,
    cmap='viridis',
    share_colorbar=True,
)

## Optional: Recent Mergers Only or Save to File

Uncomment and adjust this cell if you want the color scale clipped to a specific lookback time, or if you want to save the static figure.

In [ ]:
# fig, axes, matched_by_panel = plot_run_time_since_merger_grid(
#     run=8,
#     expts=range(9, 14),
#     merger_types=('Mini', 'Minor', 'Major'),
#     max_time_gyr=3.0,
#     vmin=0.0,
#     vmax=3.0,
#     cmap='viridis',
#     share_colorbar=True,
#     save_path='run8_expts9_13_time_since_merger.png',
# )

## Optional: Batch Runs

Uncomment this cell to generate the same continuous time-since-merger grid for several runs.

In [ ]:
# for run in [2, 3, 4, 5, 6, 7, 8]:
#     fig, axes, matched_by_panel = plot_run_time_since_merger_grid(
#         run=run,
#         expts=range(9, 14) if run == 8 else range(1, 9),
#         merger_types=('Mini', 'Minor', 'Major'),
#         alpha_background=0.35,
#         s_background=1,
#         alpha_merger=0.85,
#         s_merger=6,
#         cmap='viridis',
#         share_colorbar=True,
#         save_path=f'run{run}_time_since_merger_by_type.png',
#     )